In [1]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path

from joblib import load

# add project root
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

from src.data_enrichment import get_features

In [2]:
artifact = load("lr_tuned_pipeline.pkl")
pipeline = artifact["pipeline"]
best_threshold = artifact["threshold"]
model_features = artifact["feature_cols"]

print("Loaded tuned LR model.")
print("Threshold:", best_threshold)

Loaded tuned LR model.
Threshold: 0.9990214103801881


In [3]:

df_feats, _ = get_features("../data/raw")

df_2025 = df_feats[
    (df_feats["season_end_year"] == 2025) &
    (df_feats["minutes_played"] >= 100)
].copy()

print("2025 dataset shape:", df_2025.shape)



2025 dataset shape: (1575, 79)


In [4]:
X_2025 = df_2025[model_features].copy()

In [5]:

proba_2025 = pipeline.predict_proba(X_2025)[:, 1]

df_2025["proba_win"] = proba_2025
df_2025["predicted_winner"] = (df_2025["proba_win"] >= best_threshold).astype(int)

In [8]:

top_candidates = (
    df_2025
    .sort_values("proba_win", ascending=False)
    .loc[:, ["player_id", "player_name", "season_end_year",
         "minutes_played", "proba_win"]]
    .head(10)
)

print("\n🏆 TOP 10 Ballon d’Or Candidates for 2025:")
print(top_candidates.to_string(index=False))




🏆 TOP 10 Ballon d’Or Candidates for 2025:
player_id                player_name  season_end_year  minutes_played  proba_win
   411295          Raphinha (411295)           2025.0           252.0   0.987769
   288230   Ousmane Dembélé (288230)           2025.0           229.0   0.050650
    27992        Luka Modrić (27992)           2025.0           910.0   0.005181
    38253 Robert Lewandowski (38253)           2025.0           189.0   0.001331
   132098        Harry Kane (132098)           2025.0           194.0   0.000143
   197838       Jamie Vardy (197838)           2025.0           315.0   0.000006
    64399     Vicente Guaita (64399)           2025.0          3060.0   0.000003
   554903     Mateo Retegui (554903)           2025.0           287.0   0.000003
    23951     Steve Mandanda (23951)           2025.0          1486.0   0.000001
    17259       Manuel Neuer (17259)           2025.0          2803.0   0.000001
